# S5.3 — Streaming Responses (SSE)

Interactive verification of the `/ask` endpoint with SSE streaming and non-streaming modes.

**Spec**: `specs/spec-S5.3-streaming-responses/spec.md`

In [1]:
import sys
sys.path.insert(0, "../..")

from src.schemas.api.ask import AskRequest, AskResponse
from src.services.rag.models import SourceReference

# Verify schema imports
req = AskRequest(query="What are transformers?")
assert req.stream is True
assert req.query == "What are transformers?"
print(f"AskRequest: query={req.query!r}, stream={req.stream}, top_k={req.top_k}")
print("\n\u2713 AskRequest schema works")

AskRequest: query='What are transformers?', stream=True, top_k=None

✓ AskRequest schema works


In [2]:
# Test validation
from pydantic import ValidationError

try:
    AskRequest(query="")
    assert False, "Should have raised"
except ValidationError as e:
    print(f"Empty query rejected: {e.error_count()} error(s)")

try:
    AskRequest(query="   ")
    assert False, "Should have raised"
except ValidationError:
    print("Whitespace-only query rejected")

try:
    AskRequest(query="test", temperature=3.0)
    assert False, "Should have raised"
except ValidationError:
    print("Temperature > 2.0 rejected")

print("\n\u2713 Request validation works")

Empty query rejected: 1 error(s)
Whitespace-only query rejected
Temperature > 2.0 rejected

✓ Request validation works


In [3]:
# Test AskResponse construction
resp = AskResponse(
    answer="Transformers use self-attention [1].",
    sources=[SourceReference(index=1, arxiv_id="1706.03762", title="Attention Is All You Need",
                            authors=["Vaswani"], arxiv_url="https://arxiv.org/abs/1706.03762")],
    query="What are transformers?",
)
assert resp.answer.startswith("Transformers")
assert len(resp.sources) == 1
assert resp.sources[0].arxiv_url.startswith("https://arxiv.org/abs/")
print(f"Answer: {resp.answer}")
print(f"Sources: {[s.title for s in resp.sources]}")
print("\n\u2713 AskResponse model works")

Answer: Transformers use self-attention [1].
Sources: ['Attention Is All You Need']

✓ AskResponse model works


In [4]:
# Test SSE endpoint via ASGI transport
import json
from unittest.mock import AsyncMock
from httpx import ASGITransport, AsyncClient
from src.services.rag.models import RAGResponse, LLMMetadata, RetrievalMetadata


def _make_source(i, arxiv_id):
    return SourceReference(index=i, arxiv_id=arxiv_id, title=f"Paper {i}",
                          authors=["Author"], arxiv_url=f"https://arxiv.org/abs/{arxiv_id}")


async def fake_stream(*a, **kw):
    sources = [_make_source(1, "1706.03762")]
    yield "Hello "
    yield "world [1]."
    yield f"\n\n[SOURCES]{json.dumps([s.model_dump() for s in sources])}"


mock_rag = AsyncMock()
mock_rag.aquery_stream = fake_stream
mock_rag.aquery.return_value = RAGResponse(
    answer="Hello world [1].",
    sources=[_make_source(1, "1706.03762")],
    query="test",
)

from src.dependency import get_rag_chain
from src.main import create_app

app = create_app()
app.dependency_overrides[get_rag_chain] = lambda: mock_rag

# Streaming test
async with AsyncClient(transport=ASGITransport(app=app), base_url="http://test") as client:
    resp = await client.post("/api/v1/ask", json={"query": "test"})

assert "text/event-stream" in resp.headers["content-type"]
print("SSE Response:")
print(resp.text[:500])
print(f"\nContent-Type: {resp.headers['content-type']}")
print("\n\u2713 SSE streaming endpoint works")

SSE Response:
event: token
data: {"text": "Hello "}

event: token
data: {"text": "world [1]."}

event: sources
data: [{"index": 1, "arxiv_id": "1706.03762", "title": "Paper 1", "authors": ["Author"], "arxiv_url": "https://arxiv.org/abs/1706.03762", "chunk_text": "", "score": 0.0}]

event: done
data: {}



Content-Type: text/event-stream; charset=utf-8

✓ SSE streaming endpoint works


In [5]:
# Non-streaming test
async with AsyncClient(transport=ASGITransport(app=app), base_url="http://test") as client:
    resp = await client.post("/api/v1/ask", json={"query": "test", "stream": False})

assert "application/json" in resp.headers["content-type"]
data = resp.json()
print(f"Answer: {data['answer']}")
print(f"Sources: {len(data['sources'])} paper(s)")
print(f"Query: {data['query']}")
assert data["sources"][0]["arxiv_url"].startswith("https://arxiv.org/abs/")
print("\n\u2713 Non-streaming JSON endpoint works")

Answer: Hello world [1].
Sources: 1 paper(s)
Query: test

✓ Non-streaming JSON endpoint works


In [6]:
# Parse and verify SSE events
async with AsyncClient(transport=ASGITransport(app=app), base_url="http://test") as client:
    resp = await client.post("/api/v1/ask", json={"query": "test"})

events = []
current_event = current_data = None
for line in resp.text.split("\n"):
    if line.startswith("event: "):
        current_event = line[7:]
    elif line.startswith("data: "):
        current_data = line[6:]
    elif line == "" and current_event:
        events.append({"event": current_event, "data": current_data})
        current_event = current_data = None

event_types = [e["event"] for e in events]
print(f"Event types: {event_types}")

assert "token" in event_types, "Missing token events"
assert "sources" in event_types, "Missing sources event"
assert "done" in event_types, "Missing done event"
assert event_types[-1] == "done", "done should be last"

# Verify sources contain arXiv links
sources_event = [e for e in events if e["event"] == "sources"][0]
sources = json.loads(sources_event["data"])
for s in sources:
    assert s["arxiv_url"].startswith("https://arxiv.org/abs/")
    print(f"  Source: {s['title']} — {s['arxiv_url']}")

print(f"\n\u2713 SSE event format verified ({len(events)} events)")

Event types: ['token', 'token', 'sources', 'done']
  Source: Paper 1 — https://arxiv.org/abs/1706.03762

✓ SSE event format verified (4 events)
